# RecoMart Streaming Data Generator (Single Batch)
## User Interaction Events - Job Scheduled

This notebook generates **one batch of streaming user interaction data** per execution.

**Architecture:**
* Generates ONE micro-batch of 50 user events per run
* Writes timestamped CSV files to volume
* Designed to be scheduled via Databricks Job (every 1 minute)
* Works with streaming ingestion job to create continuous pipeline

**Output:** `/Volumes/workspace/recomart_source_data/raw/streaming/user_interactions/`

**Usage:**
* Run manually to generate one batch
* OR schedule via job for continuous streaming (recommended)
* Pair with task2_streaming_ingestion in the same job

In [0]:
# ══════════════════════════════════════════════════════════════════════════════
# Configuration
# ══════════════════════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import random
import uuid
from datetime import datetime, timedelta
import time

# ── STREAMING CONFIGURATION ───────────────────────────────────────────────────
STREAM_OUTPUT_PATH = "/Volumes/workspace/recomart_source_data/raw/streaming/user_interactions"

# Events per micro-batch (every 30 seconds)
EVENTS_PER_BATCH = 50

# Interval between batches (seconds)
BATCH_INTERVAL_SECONDS = 30

# Total runtime (set to 0 for infinite, or number of minutes)
RUN_DURATION_MINUTES = 0  # 0 = run forever until stopped

# Data generation parameters
NUM_USERS = 1000
NUM_ITEMS = 500

# Seed for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

print("✓ Streaming configuration loaded")
print(f"  Output path: {STREAM_OUTPUT_PATH}")
print(f"  Events per batch: {EVENTS_PER_BATCH}")
print(f"  Batch interval: {BATCH_INTERVAL_SECONDS} seconds")
print(f"  Runtime: {'Infinite (until stopped)' if RUN_DURATION_MINUTES == 0 else f'{RUN_DURATION_MINUTES} minutes'}")

In [0]:
# ══════════════════════════════════════════════════════════════════════════════
# Helper Functions
# ══════════════════════════════════════════════════════════════════════════════

def generate_interaction_batch(batch_size: int) -> pd.DataFrame:
    """
    Generate a micro-batch of user interaction events.
    
    Args:
        batch_size: Number of events to generate
    
    Returns:
        DataFrame with interaction events
    """
    # Event type distribution (realistic e-commerce funnel)
    event_types = ['view', 'click', 'add_to_cart', 'purchase']
    event_weights = [0.60, 0.25, 0.10, 0.05]
    
    # Device distribution
    devices = ['mobile', 'desktop', 'tablet']
    device_weights = [0.60, 0.30, 0.10]
    
    # Traffic sources
    utm_sources = ['google', 'facebook', 'instagram', 'email', 'direct', 'twitter']
    
    interaction_data = []
    for _ in range(batch_size):
        event_id = str(uuid.uuid4())[:8]
        user_id = f"U{random.randint(1, NUM_USERS):04d}"
        item_id = f"I{random.randint(1, NUM_ITEMS):04d}"
        event_type = random.choices(event_types, weights=event_weights)[0]
        
        # Current timestamp with some jitter (±5 seconds)
        jitter = random.randint(-5, 5)
        timestamp = datetime.now() + timedelta(seconds=jitter)
        
        device = random.choices(devices, weights=device_weights)[0]
        session_id = str(uuid.uuid4())[:12]
        utm_source = random.choice(utm_sources)
        
        interaction_data.append({
            'event_id': event_id,
            'user_id': user_id,
            'item_id': item_id,
            'event_type': event_type,
            'timestamp': timestamp.strftime('%Y-%m-%d %H:%M:%S'),
            'device': device,
            'session_id': session_id,
            'utm_source': utm_source
        })
    
    return pd.DataFrame(interaction_data)

def write_streaming_batch(df: pd.DataFrame, output_path: str) -> str:
    """
    Write a micro-batch to the streaming directory.
    
    Args:
        df: DataFrame to write
        output_path: Base output directory
    
    Returns:
        Path to written file
    """
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")[:19]  # Include microseconds
    filename = f"interactions_{timestamp}.csv"
    full_path = f"{output_path}/{filename}"
    
    # Convert to Spark and write
    spark_df = spark.createDataFrame(df)
    temp_path = f"{output_path}/temp_{timestamp}"
    spark_df.coalesce(1).write.mode('overwrite').option('header', 'true').csv(temp_path)
    
    # Move the CSV file
    temp_files = dbutils.fs.ls(temp_path)
    csv_file = [f for f in temp_files if f.name.endswith('.csv')][0]
    dbutils.fs.mv(csv_file.path, full_path)
    dbutils.fs.rm(temp_path, recurse=True)
    
    return full_path

print("✓ Helper functions loaded")

In [0]:
# ══════════════════════════════════════════════════════════════════════════════
# Initialize Streaming Directory
# ══════════════════════════════════════════════════════════════════════════════

try:
    # Ensure streaming directory exists
    dbutils.fs.mkdirs(STREAM_OUTPUT_PATH)
    print(f"✓ Streaming directory ready: {STREAM_OUTPUT_PATH}")
    
    # List existing files
    existing_files = dbutils.fs.ls(STREAM_OUTPUT_PATH)
    if existing_files:
        print(f"  Found {len(existing_files)} existing files")
    else:
        print("  Directory is empty - ready for new stream")
        
except Exception as e:
    print(f"⚠️  Directory setup: {e}")

In [0]:
# ══════════════════════════════════════════════════════════════════════════════
# Streaming Generator - Single Batch
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("GENERATING STREAMING DATA (SINGLE BATCH)")
print("="*80)
print(f"Execution time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Generating {EVENTS_PER_BATCH} events...")
print("-" * 80)

start_time = datetime.now()

# Generate ONE micro-batch
df_batch = generate_interaction_batch(EVENTS_PER_BATCH)

# Write to streaming directory
file_path = write_streaming_batch(df_batch, STREAM_OUTPUT_PATH)

# Event distribution in this batch
event_counts = df_batch['event_type'].value_counts().to_dict()

# Calculate stats
elapsed = (datetime.now() - start_time).total_seconds()

print(f"✓ Batch generated successfully")
print(f"  Events: {EVENTS_PER_BATCH:3d}")
print(f"  File: {file_path.split('/')[-1]}")
print(f"  Generation time: {elapsed:.2f} seconds")
print(f"  Event types:")
print(f"    - Views:         {event_counts.get('view', 0):3d} ({event_counts.get('view', 0)/EVENTS_PER_BATCH*100:.0f}%)")
print(f"    - Clicks:        {event_counts.get('click', 0):3d} ({event_counts.get('click', 0)/EVENTS_PER_BATCH*100:.0f}%)")
print(f"    - Add to cart:   {event_counts.get('add_to_cart', 0):3d} ({event_counts.get('add_to_cart', 0)/EVENTS_PER_BATCH*100:.0f}%)")
print(f"    - Purchases:     {event_counts.get('purchase', 0):3d} ({event_counts.get('purchase', 0)/EVENTS_PER_BATCH*100:.0f}%)")

print("\n" + "="*80)
print("✅ BATCH GENERATION COMPLETE")
print("="*80)
print(f"\nOutput location: {STREAM_OUTPUT_PATH}")
print("\n💡 TIP: Schedule this notebook with task2_streaming_ingestion")
print("    in a Databricks Job (every 1 minute) for continuous streaming.")
print("="*80)